# Notebook 02 — Baseline B1 tối ưu pipeline trên Kaggle T4 ×2

Notebook xác minh output NB1 rồi đọc WAV trực tiếp trong ZIP shard tại `/kaggle/input`.
Không copy hoặc giải nén audio vào `/kaggle/working`. Notebook này chỉ chạy **B1 = ViP-VL
+ Linear CTC**, seed 42, fixed budget 20 epoch. Mỗi Save Version chạy một số epoch nguyên vẹn
rồi resume ở session kế tiếp. Protocol v2 dùng 1 worker/rank, prefetch, pinned memory,
microbatch 4 và accumulation 4 để giữ effective batch 32. Bật Internet để lấy checkpoint
và `global_cmvn` đúng revision đã pin.


In [ ]:
# 1. Kiểm tra đúng 2 GPU
!nvidia-smi
import torch
assert torch.cuda.device_count() == 2, f"NB2 yêu cầu 2 GPU, hiện có {torch.cuda.device_count()}"
print([torch.cuda.get_device_name(i) for i in range(2)])


In [ ]:
# 2. Nạp source và cài dependencies
%cd /kaggle/working
%env HF_HOME=/kaggle/working/.cache/huggingface
%env HF_DATASETS_CACHE=/kaggle/working/.cache/huggingface/datasets
%env TRANSFORMERS_CACHE=/kaggle/working/.cache/huggingface/hub
%env TORCH_HOME=/kaggle/working/.cache/torch

import hashlib
import os
import shutil
import subprocess
import sys
import zipfile
from pathlib import Path

work_dir = Path("/kaggle/working/vi_asr_project")
preferred_zip = Path("/kaggle/input/vi-asr-code/vi_asr_project.zip")
preferred_project = Path("/kaggle/input/vi-asr-code/vi_asr_project")
zip_candidates = sorted(Path("/kaggle/input").rglob("vi_asr_project.zip"))
project_candidates = sorted(
    path.parent for path in Path("/kaggle/input").rglob("vi_asr_project/pyproject.toml")
)
dataset_projects = [path for path in project_candidates if "vi-asr-code" in path.parts]
dataset_zips = [path for path in zip_candidates if "vi-asr-code" in path.parts]
handoff_projects = [
    path for path in project_candidates
    if (path.parent / "artifacts" / "vi_asr_handoff.json").is_file()
]

source_path = None
source_kind = None
if (preferred_project / "pyproject.toml").is_file():
    # Kaggle thường bung ZIP đã upload thành thư mục read-only này.
    source_path, source_kind = preferred_project, "directory"
elif len(dataset_projects) == 1:
    # Kaggle image mới mount theo /kaggle/input/datasets/<owner>/<slug>/...
    source_path, source_kind = dataset_projects[0], "directory"
elif len(dataset_projects) > 1:
    raise RuntimeError(f"Có nhiều thư mục source trong dataset vi-asr-code: {dataset_projects}")
elif preferred_zip.is_file():
    source_path, source_kind = preferred_zip, "zip"
elif len(dataset_zips) == 1:
    source_path, source_kind = dataset_zips[0], "zip"
elif len(dataset_zips) > 1:
    raise RuntimeError(f"Có nhiều source ZIP trong dataset vi-asr-code: {dataset_zips}")
elif len(zip_candidates) == 1:
    source_path, source_kind = zip_candidates[0], "zip"
elif len(zip_candidates) > 1:
    raise RuntimeError(f"Có nhiều source ZIP, chỉ giữ một bản: {zip_candidates}")
elif len(handoff_projects) == 1:
    source_path, source_kind = handoff_projects[0], "directory"
elif not handoff_projects and len(project_candidates) == 1:
    source_path, source_kind = project_candidates[0], "directory"
else:
    raise FileNotFoundError(
        "Không xác định được duy nhất source project. Hãy attach dataset vi-asr-code "
        "hoặc output Notebook 01."
    )

if not (work_dir / "pyproject.toml").is_file():
    if source_kind == "zip":
        with zipfile.ZipFile(source_path, "r") as archive:
            archive.extractall("/kaggle/working")
        print("Đã giải nén source:", source_path)
    else:
        shutil.copytree(source_path, work_dir)
        print("Đã copy source:", source_path)

def sha256_source(path):
    path = Path(path)
    digest = hashlib.sha256()
    files = [path] if path.is_file() else sorted(item for item in path.rglob("*") if item.is_file())
    for item in files:
        relative = item.name if path.is_file() else item.relative_to(path).as_posix()
        digest.update(relative.encode("utf-8"))
        digest.update(b"\0")
        with item.open("rb") as stream:
            for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
                digest.update(block)
    return digest.hexdigest()

source_sha256 = sha256_source(source_path)
print({"source": str(source_path), "kind": source_kind, "sha256": source_sha256})

os.environ["PYTHONPATH"] = str(work_dir / "src")
if str(work_dir / "src") not in sys.path:
    sys.path.insert(0, str(work_dir / "src"))

%cd /kaggle/working/vi_asr_project
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements-kaggle.in"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-e", "."], check=True)

import vi_asr
print("vi_asr:", vi_asr.__file__)


In [ ]:
# 3. Xác minh và kích hoạt artifact NB1 — không giải nén audio
import subprocess, sys
assert "vi-asr-code" in source_path.parts, (
    f"NB2 từ chối source fallback ngoài dataset vi-asr-code: {source_path}"
)
subprocess.run([
    sys.executable, "scripts/kaggle_handoff.py", "restore",
    "--input-root", "/kaggle/input",
    "--working-root", "/kaggle/working",
    "--runtime-paths", "configs/paths/kaggle_runtime.yaml",
], check=True)


In [ ]:
# 4. Preflight: data_root phải nằm trong Input và mọi shard phải hợp lệ
import json
from pathlib import Path
from scripts.kaggle_handoff import validate_audio_shards

runtime = json.loads(Path("configs/paths/kaggle_runtime.yaml").read_text(encoding="utf-8"))
data_root = Path(runtime["paths"]["data_root"])
assert str(data_root).startswith("/kaggle/input/"), data_root
assert not Path("/kaggle/working/artifacts/prepared/audio").exists()

verified = validate_audio_shards(data_root, verify_hashes=False)  # SHA đã kiểm tra ở cell 3
print(json.dumps(verified, indent=2))
assert verified["split_counts"] == {"train": 15023, "valid": 1900, "test": 2026}
assert Path("artifacts/tokenizer/bpe_1024.model").is_file()

subprocess.run([
    sys.executable, "-m", "vi_asr", "audit",
    "--config", "configs/base.yaml",
    "--paths", "configs/paths/kaggle_runtime.yaml",
    "--report-dir", "/kaggle/working/preflight_audit",
], check=True)


In [ ]:
# 5. Smoke tùy chọn — chỉ cần bật ở session B1 đầu tiên
RUN_SMOKE = False
if RUN_SMOKE:
    subprocess.run([
        sys.executable, "-m", "vi_asr", "smoke",
        "--config", "configs/base.yaml",
        "--paths", "configs/paths/kaggle_runtime.yaml",
    ], check=True)
else:
    print("Bỏ qua smoke đã PASS trước đó; bắt đầu đường chạy B1 cố định.")


In [ ]:
# 6. Protocol B1 đã khóa — không đổi giữa các session
RUN_NAME = "b1_ctc_v2"
SEED = 42
TOTAL_EPOCHS = 20
EPOCHS_THIS_SESSION = 3         # giảm còn 1–2 nếu gần giới hạn thời gian Kaggle
RESUME_CHECKPOINT = "auto"      # auto | None | đường dẫn checkpoint cụ thể

assert EPOCHS_THIS_SESSION > 0
print({
    "run": RUN_NAME,
    "variant": "configs/variants/b1.yaml",
    "seed": SEED,
    "total_epochs": TOTAL_EPOCHS,
    "epochs_this_session": EPOCHS_THIS_SESSION,
    "resume": RESUME_CHECKPOINT,
})


In [ ]:
# 7. Chạy/resume baseline B1; subprocess lỗi sẽ dừng notebook ngay
import hashlib
import json
import shutil
import subprocess
from pathlib import Path
import torch

common = [
    "--config", "configs/base.yaml",
    "--profile", "configs/profiles/t4x2_fast.yaml",
    "--paths", "configs/paths/kaggle_runtime.yaml",
]

command = [
    "torchrun", "--standalone", "--nproc_per_node=2", "-m", "vi_asr", "train",
    *common,
    "--variant", "configs/variants/b1.yaml",
    "--run-dir", f"/kaggle/working/runs/{RUN_NAME}",
    "--seed", str(SEED),
    "--epochs-this-session", str(EPOCHS_THIS_SESSION),
]

resume_path = None
if RESUME_CHECKPOINT == "auto":
    candidates = sorted(
        path for path in Path("/kaggle/input").rglob("checkpoint_latest.pt")
        if path.parent.name == RUN_NAME
    )
    if len(candidates) > 1:
        raise RuntimeError(f"Chỉ attach output session B1 mới nhất; đang có: {candidates}")
    resume_path = candidates[0] if candidates else None
elif RESUME_CHECKPOINT:
    resume_path = Path(RESUME_CHECKPOINT)

start_epoch = 1
if resume_path is not None:
    assert resume_path.is_file(), resume_path
    resume_meta = torch.load(resume_path, map_location="cpu", weights_only=False)
    assert resume_meta.get("config", {}).get("experiment", {}).get("name") == "b1_ctc"
    assert resume_meta["config"]["experiment"].get("protocol_version") == "b1_repro_v2"
    assert int(resume_meta["config"]["experiment"]["seed"]) == SEED
    start_epoch = int(resume_meta["epoch"]) + 1
    assert start_epoch <= TOTAL_EPOCHS, "B1 đã đủ 20 epoch; chuyển sang Notebook 03."
    command += ["--resume", str(resume_path)]
    print(f"Resume epoch {start_epoch}/{TOTAL_EPOCHS} từ:", resume_path)
else:
    print("Bắt đầu B1 mới từ pretrained ViP-VL, epoch 1/20.")

def sha256(path):
    digest = hashlib.sha256()
    with Path(path).open("rb") as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b""):
            digest.update(block)
    return digest.hexdigest()

protocol = {
    "run": RUN_NAME,
    "variant": "b1",
    "protocol_version": "b1_repro_v2",
    "profile": "configs/profiles/t4x2_fast.yaml",
    "seed": SEED,
    "total_epochs": TOTAL_EPOCHS,
    "session_start_epoch": start_epoch,
    "epochs_this_session": EPOCHS_THIS_SESSION,
    "source_path": str(source_path),
    "source_kind": source_kind,
    "source_sha256": source_sha256,
    "data_root": str(data_root),
    "audio_files": verified["files"],
    "split_counts": verified["split_counts"],
}
protocol_path = Path(f"/kaggle/working/runs/{RUN_NAME}/notebook_protocol.json")
protocol_path.parent.mkdir(parents=True, exist_ok=True)
protocol_path.write_text(json.dumps(protocol, indent=2, ensure_ascii=False), encoding="utf-8")
print(json.dumps(protocol, indent=2, ensure_ascii=False))

free_gib = shutil.disk_usage("/kaggle/working").free / 1024**3
print("Free working before training:", round(free_gib, 2), "GiB")
assert free_gib >= 8.0, "Working còn dưới 8 GiB; hãy xóa output/cache không cần thiết"
print("Command:", " ".join(command), flush=True)
training_env = os.environ.copy()
training_env["PYTHONUNBUFFERED"] = "1"
subprocess.run(command, check=True, env=training_env)


In [ ]:
# 8. Kiểm tra output và dọn cache trước Save Version
import os
import shutil
from pathlib import Path

run_root = Path("/kaggle/working/runs")
for checkpoint in sorted(run_root.rglob("*.pt")):
    stat = checkpoint.stat()
    print(f"{checkpoint}: {stat.st_size / 1024**3:.2f} GiB | inode={getattr(stat, 'st_ino', 'n/a')}")

cache_root = Path("/kaggle/working/.cache")
if cache_root.exists():
    shutil.rmtree(cache_root)

# Source luôn lấy lại từ NB1 input ở session kế tiếp; không lưu bản giải nén trong output NB2.
os.chdir("/kaggle/working")
project_copy = Path("/kaggle/working/vi_asr_project")
if project_copy.exists():
    shutil.rmtree(project_copy)
print("Working free before Save Version:", round(shutil.disk_usage("/kaggle/working").free / 1024**3, 2), "GiB")

latest = run_root / RUN_NAME / "checkpoint_latest.pt"
best = run_root / RUN_NAME / "checkpoint_best.pt"
assert latest.is_file() and best.is_file(), (latest, best)
meta = torch.load(latest, map_location="cpu", weights_only=False)
finished_epoch = int(meta["epoch"])
assert 1 <= finished_epoch <= TOTAL_EPOCHS
print(f"B1 đã hoàn tất epoch {finished_epoch}/{TOTAL_EPOCHS}; best epoch={meta['metrics']['best_epoch']}")
if finished_epoch < TOTAL_EPOCHS:
    print("PASS SESSION: Save Version, session sau chỉ attach output B1 mới nhất và resume auto.")
else:
    print("B1 TRAINING COMPLETE: Save Version rồi chuyển sang Notebook 03 đánh giá.")
